# Lab 13–15: Pipeline Telemetry & Prometheus Observability

**Z5008 Big Data Lab (MTech Core) · IIT Madras Zanzibar**

Tail latency quantiles (p50, p95, p99) and Prometheus metric formatting.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inyalala/iitmz-teaching-site/blob/main/courses/z5008/labs/BD05_Pipeline_Telemetry_Observability.ipynb)

Run cells top to bottom (`Runtime > Run all`). Spark steps include pure-Python fallbacks when executed outside a Spark/JVM container.

## 1. Tail Latency Percentiles (`p50`, `p95`, `p99`) & Prometheus Exposition

Averages hide tail stalls in distributed pipelines; tracking `p95`/`p99` stage latencies alongside ok/error counters enables Prometheus + Grafana SLA alerting.

In [1]:
import math
import random


def percentile(vals, p):
    s = sorted(vals)
    k = (len(s) - 1) * (p / 100.0)
    f, c = math.floor(k), math.ceil(k)
    return s[int(k)] if f == c else s[f] + (k - f) * (s[c] - s[f])


rng = random.Random(5008)
latencies = [rng.lognormvariate(3.0, 0.45) for _ in range(400)]
p50, p95, p99 = percentile(latencies, 50), percentile(latencies, 95), percentile(latencies, 99)
print(f"p50={p50:.2f} ms | p95={p95:.2f} ms | p99={p99:.2f} ms")
assert p50 < p95 < p99

print("\nPrometheus scrape preview:")
print(f'pipeline_records_total{{job="lakehouse_etl",status="ok"}} 10000')
print(f'pipeline_stage_latency_ms{{job="lakehouse_etl",quantile="0.95"}} {p95:.2f}')


p50=19.65 ms | p95=43.74 ms | p99=50.20 ms

Prometheus scrape preview:
pipeline_records_total{job="lakehouse_etl",status="ok"} 10000
pipeline_stage_latency_ms{job="lakehouse_etl",quantile="0.95"} 43.74
